In [5]:
import os
import pandas as pd
from pathlib import Path
from omegaconf import OmegaConf

from annotate import *

In [6]:
cfg = OmegaConf.load("../../config/default.yaml")

# Resolve paths
project_root = Path("../..").resolve()

# Define paths based on the configuration
hf_data = project_root / cfg.data.paths.hf
labels_dir = project_root / cfg.data.paths.base

In [7]:
# Load labels into a DataFrame
df = pd.DataFrame()
splits = ['dev_seen', 'dev_unseen', 'test_seen', 'test_unseen', 'train']
for split in splits:
    df_tmp = pd.read_json(f'{hf_data}/{split}.jsonl', lines=True)
    df_tmp['split'] = split
    df = pd.concat([df, df_tmp])

In [8]:
# Verify if images exist in the hf_data directory
hf_images = os.listdir(hf_data / "img")
df['image_found'] = df['img'].str.lstrip('img/').isin(hf_images)

In [9]:
print(f"Total number of lables: {len(df)}")

# Drop rows where the image is not found
print(f"Number of dropped rows: {len(df[df['image_found'] == False])}")
df = df[df['image_found'] == True]

# Drop duplicate images
print(f"Number of dropped duplicated images: {df['img'].duplicated().sum()}")
df = df.drop_duplicates(subset=['img'], keep='first')

Total number of lables: 12540
Number of dropped rows: 2557
Number of dropped duplicated images: 319


In [10]:
# Sample 20% of the DataFrame, keeping split proportions, with a fixed seed
sampled_df = df.groupby('split').sample(frac=0.2, random_state=42, replace=False)

In [ ]:
annotate_data(sampled_df, labels_dir / 'labels.jsonl', 'Nils')